# HippoVoice — Full New-Systems Benchmark (WeightEdit + Track 2 Audio, Kaggle T4)

One file, two things this project hadn't produced a real number for yet:

1. **WeightEdit** (Sections 1-6): `baselines/weight_edit_baseline.py`'s
   real, non-mocked ROME/MEMIT-on-GPT-2-XL path.
2. **HippoAudio / Track 2** (Sections 7-9): `HippoAudioPipeline` +
   `GeminiLiveAudioModel`'s real, non-mocked audio-to-audio path.

Both run through the exact same LoCoMo harness pieces (dataset loading,
conversation flattening, F1 scoring) that HippoVoice/Mem0-style/A-MEM-
style/NaiveRAG already ran through on `colab.ipynb` — so both numbers are
genuinely comparable to those, not isolated demos. Section 10 combines
everything into one results file next to those existing numbers.

**Scope, stated plainly**: this notebook does NOT re-run HippoVoice/Mem0/
A-MEM/NaiveRAG — those already have real results from `colab.ipynb`
(HippoVoice: 24.1% avg F1 over 1540 questions, confirmed 2026-07-11; see
`BUGS.md`), and nothing about their own code changed, so re-running them
isn't needed. This file only adds the two pieces that were still missing.

**What WeightEdit tests**: instead of storing facts externally and
retrieving them at generation time (every other system in this project),
can directly editing a model's weights with each new fact work as well or
better? See `baselines/weight_edit_baseline.py`'s module docstring for why
nobody does this in production — but this actually checks rather than
assuming. Expected outcome, stated up front: ROME/MEMIT were validated on
single, discrete edits, not hundreds of accumulating facts — published
work shows sequential edits degrading well before LoCoMo's 369-689
turns/conversation. Seeing that degradation for real here is a legitimate
result, not a failed benchmark.

**What HippoAudio adds over what already existed**: before this notebook,
Track 2's only evidence was a hand-run 3-turn demo (a fact stated in turn
1, correctly recalled in turn 3 — see `BUGS.md`) — real, genuinely
confirmed memory *conditioning*, but not an F1 number comparable to
anything else in this project. Section 8 produces that number, at
(intentionally) small scale first, same discipline as WeightEdit's.

**Run this top to bottom once, in order.** Sections 3b and 7c are
deliberate, cheap sanity checks before spending real time/API-quota on
either benchmark — this project's established discipline (see `BUGS.md`,
the extraction-prompt saga) is to validate small before validating
expensive.

**Logging**: every section from Section 2 onward runs inside a `step(...)`
block (defined in Section 1b) that writes timestamped start/done/FAILED
lines to a log file under `/kaggle/working`, flushed to disk immediately
after every line — not buffered until the cell finishes. A failed step is
logged and recorded, not re-raised immediately, so later cells (crucially,
Sections 6/9/10's Save Results) still run and produce real output instead
of a failed commit leaving nothing behind (see `BUGS.md` for a real,
previously-hit case of exactly that on `colab.ipynb`). Section 10 is the
one place that re-raises, at the very end, if anything failed — so the
commit still ends up correctly marked failed, but only after every
artifact that could be saved, was.

## 1. Clone hippovoice + GPU check

In [ ]:
# Same pattern as colab.ipynb's Step 2 -- run after selecting
# Accelerator: GPU T4 x2 in Kaggle's notebook settings (top right).
# Private repo: add a GitHub PAT to Kaggle Secrets as GH_TOKEN, then enable it.
#
# Uses subprocess with check=True, not !magic -- !magic swallows a non-zero
# exit code silently (confirmed for real elsewhere in this notebook, see
# Section 1b's run_shell docstring); check=True raises CalledProcessError
# instead, which is the right behavior for this specific cell: nothing
# later can possibly work if the repo itself never actually cloned, so a
# loud immediate failure here (a normal red Jupyter error, since the
# log()/step() infra below doesn't exist yet at this point) is correct,
# not something to swallow and continue past.

import os, sys, subprocess

if os.path.exists('/kaggle/working'):
    REPO_DIR = '/kaggle/working/hippovoice'
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GH_TOKEN')
        CLONE_URL = f'https://{token}@github.com/shivansh193/hippovoice.git'
    except Exception:
        CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'
        print('No GH_TOKEN secret found -- will work for public repos only')
else:
    REPO_DIR = '/content/hippovoice'
    CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', CLONE_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    commit = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h'], text=True).strip()
except Exception:
    commit = 'unknown'
print(f'commit [{commit}]')
print('Ready.')

In [ ]:
import torch

# Confirmed the hard way on this project's other Kaggle work (see BUGS.md,
# colab.ipynb): checking immediately after clone fails in seconds instead
# of hours into a run that turns out to be running on CPU. ROME/MEMIT edits
# involve real gradient descent (v_num_grad_steps=20 per edit) -- CPU would
# make even the single-edit sanity check in 3b painfully slow, let alone a
# full benchmark.
assert torch.cuda.is_available(), (
    'No GPU detected -- set Accelerator to "GPU T4 x2" in Kaggle Settings '
    '(top-right of the notebook editor) before running anything below.'
)
print(f'GPU OK: {torch.cuda.get_device_name(0)}')

## 1b. Logging setup

`log(msg)` prints AND appends to a log file, flushed + fsynced immediately
(not buffered) so a crash doesn't lose the line that would have explained
it. `step(name)` wraps a block of work: logs START/DONE automatically, and
on any exception logs FAILED with the full traceback -- and, deliberately,
does NOT re-raise. A "Save & Run All (Commit)" job stops at the first
uncaught exception (confirmed the hard way on `colab.ipynb`, see
`BUGS.md` -- a crashed cell took the whole commit down with it, including
the Save Results cell that would have written the one real number from
that run), so if editing/loading fails partway through, swallowing it here
instead of re-raising is what lets Section 6 still run and write a results
file -- reporting the failure honestly rather than saying nothing at all.

Every failed step is appended to `_failures`; Section 6 writes them into
the results JSON, then deliberately re-raises at the very end if the list
is non-empty -- so the commit still ends up correctly marked failed in
Kaggle's UI, but only after every artifact (log, checkpoint, results JSON)
that could be saved, was.

`run_shell(cmd)` exists because of a real, confirmed gap: `!pip install
...` / `!git clone ...` shell-magic lines do NOT raise a Python exception
on a non-zero exit code -- IPython just prints the error and moves on. A
failed EasyEdit `pip install -r requirements.txt` was silently logged as
`DONE` on a real run because of exactly this (`step()` only catches
Python exceptions, and none was ever raised). Every shell command from
Section 2 onward runs through `run_shell` instead of `!magic` specifically
so a real failure becomes a real exception `step()` can actually catch,
with the full stdout+stderr captured into `run_log.txt` either way (not
just whatever Kaggle's own cell-output view happens to show, which can
get truncated in the log you're reading it from).

In [ ]:
import contextlib
import datetime
import subprocess
import traceback

LOG_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
LOG_PATH = f'{LOG_BASE}/run_log.txt'
_failures = []  # [(step_name, exception_repr), ...] -- checked by Section 6

def log(msg):
    line = f'[{datetime.datetime.now().isoformat(timespec="seconds")}] {msg}'
    print(line)
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
        f.flush()
        os.fsync(f.fileno())

@contextlib.contextmanager
def step(name):
    log(f'START   {name}')
    try:
        yield
    except Exception as e:
        log(f'FAILED  {name}\n{traceback.format_exc()}')
        _failures.append((name, repr(e)))
        print(f'>>> {name} FAILED -- continuing so later cells (esp. Section 6, Save Results) '
              f'still run and record this. See {LOG_PATH} for the full traceback. <<<')
    else:
        log(f'DONE    {name}')

def require(*names):
    """Confirmed for real: without this, a step whose prerequisite (editor,
    extraction_llm, audio_model, ...) never got set because an EARLIER step
    already failed and got swallowed (by design -- see step() above) only
    surfaces several cells later as a bare `NameError: name 'audio_model' is
    not defined`, with nothing pointing back at the actual root cause one
    has to scroll up and find manually. Call this as the first line inside
    a step() block for anything it depends on -- raises immediately with a
    message naming exactly what's missing and where to look."""
    missing = [n for n in names if n not in globals() or globals()[n] is None]
    if missing:
        raise RuntimeError(
            f'Missing prerequisite(s): {missing} -- an earlier step must have '
            f'failed before setting these. Check run_log.txt for the FAILED '
            f'entry that comes before this one.'
        )

def run_shell(cmd, cwd=None):
    """subprocess.run instead of !magic -- see the markdown cell above for
    why: !magic swallows non-zero exit codes silently, which is exactly
    how a real pip install failure once got logged as DONE. Logs the full
    stdout+stderr (truncated in the printed summary, complete in the log
    file) and raises RuntimeError on a non-zero exit code so step() can
    actually catch it."""
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    output = (result.stdout or '') + (result.stderr or '')
    log(f'$ {cmd}  (exit {result.returncode})\n{output}')
    print(output[-2000:])  # tail only in the cell's own live output; full text is in run_log.txt
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd} -- see {LOG_PATH} for full output')
    return result

log(f'=== New run -- log path: {LOG_PATH} ===')
print(f'Logging to {LOG_PATH}')

## 2. Clone + install EasyEdit

Not pip-installable in a way that ships its `hparams/*.yaml` files --
`ROMEHyperParams.from_hparams()` reads those directly from a cloned repo,
so cloning is required regardless of whether a PyPI `easyeditor` package
also exists.

**Confirmed on a real run**: installing EasyEdit's `requirements.txt` in
one `pip install -r ...` shot failed -- a wheel-build error for some
package partway through the ~30-package list (pins `torch==2.9.1`,
`transformers==5.5.4`, plus a long tail of vision/audio-multimodal deps
like `opencv-python`, `av`, `qwen_vl_utils`, `fairscale` that this project
doesn't need for plain ROME/MEMIT on GPT-2-XL -- EasyEdit ships those for
model types this notebook never touches). One failing package aborted the
whole install with no clear indication of which one.

**Confirmed on a real run, a second failure past the install itself**:
even after installing "successfully", `import easyeditor` later crashed
with `ImportError: cannot import name '_center' from
'numpy._core.umath'`. Root cause: Kaggle's base image already has numpy/
scipy/scikit-learn/torch/transformers preinstalled (torch already matched
to the correct CUDA build) -- EasyEdit's `==`-exact-pinned requirements.txt
force-reinstalls all of these on top of an already-running Python process.
A compiled extension (scipy/scikit-learn) that was already loaded against
the OLD numpy's C ABI, combined with a freshly-installed DIFFERENT numpy
version, breaks in exactly this way -- and torch got downgraded from
Kaggle's own 2.10.0+cu128 to EasyEdit's pinned 2.9.1 for no real benefit,
costing a ~900MB re-download in the process.

Fixed: skip installing anything already importable in the current
environment (checked via `importlib.util.find_spec`, not by trying to
parse pip's own installed-version output) rather than blindly reinstalling
every pinned version -- avoids the numpy corruption, and is faster since
foundational packages Kaggle already ships (torch, numpy, scipy,
scikit-learn, transformers, sentence-transformers, ...) never get
re-downloaded. Genuinely-missing packages (EasyEdit's vision/audio-
multimodal deps, `higher`, `hydra-core`, ...) still install normally, one
at a time, logging (not raising) on a per-package failure so one bad
package can't block the rest -- `Section 4`'s `import easyeditor` is the
real test of whether a skipped-or-failed package actually mattered.

In [ ]:
with step('clone + install EasyEdit'):
    EASYEDIT_DIR = os.path.join(os.path.dirname(REPO_DIR), 'EasyEdit')

    if os.path.exists(os.path.join(EASYEDIT_DIR, '.git')):
        run_shell(f'git -C {EASYEDIT_DIR} pull')
    else:
        run_shell(f'git clone https://github.com/zjunlp/EasyEdit.git {EASYEDIT_DIR}')

    req_path = os.path.join(EASYEDIT_DIR, 'requirements.txt')
    with open(req_path) as f:
        requirements = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]

    import importlib.util

    # pip package name -> the name you actually `import` -- only needed for
    # the ones that differ; everything else falls back to name.replace('-','_').
    _IMPORT_NAME = {
        'pyyaml': 'yaml', 'scikit-learn': 'sklearn', 'opencv-python': 'cv2',
        'hydra-core': 'hydra', 'importlib-metadata': 'importlib_metadata',
        'sentence-transformers': 'sentence_transformers',
    }

    failed_packages = []
    skipped_packages = []
    for req in requirements:
        pkg_name = req.split('==')[0].split('>=')[0].split('<')[0].strip()
        import_name = _IMPORT_NAME.get(pkg_name.lower(), pkg_name.replace('-', '_'))
        if importlib.util.find_spec(import_name) is not None:
            skipped_packages.append(req)
            log(f'  skipping {req} -- {import_name} already importable (Kaggle base image)')
            continue
        try:
            run_shell(f'pip install -q "{req}"')
        except RuntimeError:
            failed_packages.append(req)
            log(f'  (continuing past {req} -- Section 4 will show whether this actually matters)')

    run_shell('pip install -q google-genai')

    if skipped_packages:
        log(f'Skipped {len(skipped_packages)} already-available package(s): {skipped_packages}')
    if failed_packages:
        log(f'WARNING: {len(failed_packages)} EasyEdit requirement(s) failed to install: {failed_packages}')
        print(f'>>> {len(failed_packages)} package(s) failed to install: {failed_packages}')
        print('    This may or may not matter -- Section 4 (import easyeditor) will tell you for sure.')
    else:
        log('All remaining EasyEdit requirements installed successfully.')

    log(f'EasyEdit cloned to {EASYEDIT_DIR}')

## 3. Load GPT-2 XL + ROME or MEMIT (one-time -- reused across conversations)

Loaded once here, not per-conversation: `WeightEditBaseline` resets this
same editor's weights back to pristine at the start of each conversation
(see its `__init__` and the module docstring) rather than reloading the
~6GB model from HuggingFace every time, exactly mirroring how the RAG
baselines reuse one loaded LLM across conversations and only reset their
own memory store per conversation.

`METHOD` mirrors `colab.ipynb`'s `SYSTEM` toggle -- change this one line
and re-run from here to switch. Defaults to `'ROME'`: confirmed directly
from both hparams files that ROME's gpt2-xl config sets
`mom2_adjustment: false` (no covariance-statistics precompute needed)
while MEMIT's sets it `true` with `mom2_n_samples: 100000` -- a real,
unmeasured cost this notebook hasn't paid or timed yet. Get ROME's number
first; MEMIT is this same interface, not a separate notebook.

In [ ]:
METHOD = 'ROME'  # 'ROME' | 'MEMIT'

with step(f'load GPT-2 XL + {METHOD}'):
    from baselines.easyedit_weight_editor import EasyEditWeightEditor

    editor = EasyEditWeightEditor(easyedit_dir=EASYEDIT_DIR, method=METHOD)
    log('Loading GPT-2 XL (first run downloads ~6GB from HuggingFace)...')
    editor.load()
    log('Loaded.')

## 3b. Cheap sanity check -- confirm editing actually works before anything else

Deliberately not a LoCoMo question -- a well-known fact whose default
completion is predictable, so a wrong "BEFORE" or unchanged "AFTER" is
obviously a real problem rather than model uncertainty. **Read the three
printed lines before continuing**: BEFORE should be the true, un-edited
answer; AFTER-edit should reflect the new fact; AFTER-reset should return
to (approximately) the BEFORE answer. If AFTER-edit doesn't change, or
AFTER-reset doesn't revert, stop here and debug -- nothing downstream
(hundreds of sequential edits per LoCoMo conversation) will work either if
a single edit doesn't.

In [ ]:
with step('single-edit sanity check'):
    prompt = "The Eiffel Tower is located in the city of"

    before = editor.generate(prompt, max_tokens=8)
    log(f'BEFORE edit: {before!r}')

    editor.edit(prompt=prompt, subject="The Eiffel Tower", target_new="Rome")
    after_edit = editor.generate(prompt, max_tokens=8)
    log(f'AFTER edit:  {after_edit!r}')

    editor.reset()
    after_reset = editor.generate(prompt, max_tokens=8)
    log(f'AFTER reset: {after_reset!r}')

print()
print('>>> Check the three lines above before continuing:')
print('    AFTER edit should differ from BEFORE and reflect the new fact.')
print('    AFTER reset should return to (approximately) BEFORE.')
print('    If either does not hold, stop and debug -- do not proceed to Section 5.')

## 4. Gemini extraction client

Used only for extraction and edit-request conversion (turning a free-text
memory into a ROME-style prompt/subject/target_new) -- never for QA
answers, which come from the edited model itself. Same reasoning as
`llm/gemini_client.py`'s own docstring: keeps everything except the model
actually being edited API-based, no second heavy local model to load
alongside GPT-2 XL.

Add your key to Kaggle Secrets as `GEMINI_API_KEY` and enable it for this
notebook, or paste it directly below for a quick run (remove it again
afterward if you do).

In [ ]:
with step('load Gemini extraction client'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
        log('Loaded GEMINI_API_KEY from Kaggle Secrets')
    except Exception:
        if not os.environ.get('GEMINI_API_KEY'):
            log('No GEMINI_API_KEY in Kaggle Secrets or the environment -- set one before continuing:')
            print("  os.environ['GEMINI_API_KEY'] = '...'")

    from llm.gemini_client import GeminiTextLLM
    extraction_llm = GeminiTextLLM()
    log(f'Extraction LLM: {extraction_llm.model_name}')

## 5. Run WeightEditBaseline through the real LoCoMo benchmark

Same `run_locomo` harness, same F1 scoring, as every other system's LoCoMo
number in this project. `pipeline_factory` closes over the single shared
`editor` loaded in Section 3 -- `WeightEditBaseline.__init__` resets it to
pristine weights for every new conversation (validated locally against
`MockWeightEditor` in `tests/test_weight_edit_baseline.py`; Section 3b just
confirmed `editor.reset()` itself works against the real model too).

**Scope starts deliberately small.** Every extracted, editable memory is a
real ROME/MEMIT edit -- gradient descent against the actual model, not a
cheap dict write -- so per-turn cost here is categorically different from
the RAG baselines' pure-retrieval QA step. `NUM_CONVERSATIONS=1` and
`MAX_QA_PER_CONVERSATION=15` first, to get a real wall-clock/turn reading
before committing to anything close to the other baselines' full
10-conversation, ~1540-question runs. Widen both only after seeing how
long one conversation actually takes.

`run_locomo`'s own `checkpoint_path` already saves progress after each
conversation and would let a second run resume -- but at
`NUM_CONVERSATIONS=1` there's only one conversation to lose, so the
`step(...)` wrapper's job here is mainly making sure a mid-run failure
(an API error, an OOM on some later edit) still writes what happened, and
approximately where, to `run_log.txt` instead of vanishing along with the
crashed cell.

In [ ]:
from benchmarks.locomo.evaluate import run_locomo
from baselines.weight_edit_baseline import WeightEditBaseline
import time

NUM_CONVERSATIONS = 1
MAX_QA_PER_CONVERSATION = 15

def _weight_edit_factory(llm):
    return WeightEditBaseline(llm_client=llm, editor=editor)

CHECKPOINT_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
CHECKPOINT_PATH = f'{CHECKPOINT_BASE}/locomo_checkpoint_weightedit_{METHOD.lower()}.json'

locomo_result = None
_elapsed = None

with step(f'run_locomo (WeightEdit-{METHOD}, {NUM_CONVERSATIONS} conv, '
          f'{MAX_QA_PER_CONVERSATION} qa/conv)'):
    require('editor', 'extraction_llm')
    log(f'Checkpoint path: {CHECKPOINT_PATH}')
    _t0 = time.time()
    locomo_result = run_locomo(
        llm_client=extraction_llm,
        num_conversations=NUM_CONVERSATIONS,
        max_qa_per_conversation=MAX_QA_PER_CONVERSATION,
        checkpoint_path=CHECKPOINT_PATH,
        pipeline_factory=_weight_edit_factory,
        system_name=f'WeightEdit-{METHOD}',
    )
    _elapsed = time.time() - _t0

    log(f"avg F1: {locomo_result['avg_f1']:.1%}  (over {locomo_result['total']} questions)  "
        f"bins: {locomo_result['bins']}  wall_clock: {_elapsed:.0f}s")


# Guarded on locomo_result rather than assumed -- if the step above failed,
# _failures already has it and Section 6 will still save a results file
# recording that; this block just needs to not ALSO raise an unguarded
# TypeError on a None result (confirmed as a real second bug: it did,
# on the same run the chromadb import crashed, because this summary lived
# outside the with step(...) block and unconditionally subscripted
# locomo_result -- which stayed None -- stopping the whole run right after
# the failure was already correctly caught and logged).
if locomo_result is not None:
    print('=' * 60)
    print(f"WeightEdit-{METHOD} LoCoMo avg F1: {locomo_result['avg_f1']:.1%}  "
          f"(over {locomo_result['total']} questions)")
    print(f"bins: {locomo_result['bins']}")
    print(f'Wall clock: {_elapsed:.0f}s for {NUM_CONVERSATIONS} conversation(s) '
          f'-- use this to size NUM_CONVERSATIONS/MAX_QA_PER_CONVERSATION for a bigger run.')
    print('=' * 60)
    for d in sorted(locomo_result['details'], key=lambda d: -d['f1'])[:10]:
        print(f"  f1={d['f1']:.2f} cat={d['category']}  Q: {d['question']}")
        print(f"    gold={d['gold']!r}  predicted={d['predicted']!r}")
else:
    print(f'run_locomo did not complete -- see {LOG_PATH} for why. Section 6 will still save a results file.')

## 6. Save WeightEdit results (intermediate -- combined with Track 2 below in Section 10)

Written defensively: if an earlier cell failed, the variables it would
have set (`locomo_result`, `_elapsed`, `extraction_llm`) simply won't
exist -- this cell checks for that with `dir()` instead of crashing on a
`NameError`, so you always get a results JSON (and this cell's own log
entry) recording how far the run actually got, not nothing.

In [ ]:
import json
import datetime

with step('save WeightEdit results'):
    WEIGHTEDIT_RESULTS_PATH = (
        ('/kaggle/working' if os.path.exists('/kaggle/working') else '/content')
        + f'/hippovoice_results_weightedit_{METHOD.lower()}.json'
    )

    weightedit_out = {
        'timestamp': datetime.datetime.now().isoformat(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
        'system': f'WeightEdit-{METHOD}',
        'extraction_llm': extraction_llm.model_name if 'extraction_llm' in dir() else 'not reached',
        'num_conversations': globals().get('NUM_CONVERSATIONS', 'not reached'),
        'max_qa_per_conversation': globals().get('MAX_QA_PER_CONVERSATION', 'not reached'),
        'wall_clock_seconds': _elapsed if 'locomo_result' in dir() and locomo_result is not None else 'not reached',
        'locomo': (
            {
                'avg_f1': locomo_result['avg_f1'],
                'total': locomo_result['total'],
                'bins': locomo_result['bins'],
            }
            if 'locomo_result' in dir() and locomo_result is not None
            else 'not reached -- see run_log.txt for where the run stopped'
        ),
    }

    with open(WEIGHTEDIT_RESULTS_PATH, 'w') as f:
        json.dump(weightedit_out, f, indent=2)

    log(f'Saved to {WEIGHTEDIT_RESULTS_PATH}')

# Guarded the same way as Section 5's summary block: if the step above
# failed before either variable got assigned, don't compound one already-
# logged failure with a second unguarded NameError/TypeError out here.
if 'weightedit_out' in dir():
    print(f'Results: {WEIGHTEDIT_RESULTS_PATH}')
    print(json.dumps(weightedit_out, indent=2))
else:
    print(f'Saving WeightEdit results failed -- see {LOG_PATH}.')

## 7. Track 2 -- audio pipeline (HippoAudioPipeline + Gemini Live) real LoCoMo benchmark

Runs `benchmarks/locomo/evaluate_audio.py::run_locomo_audio` -- the same
LoCoMo dataset, conversation flattening, and F1 scoring as every text
system's number in this project, driving `HippoAudioPipeline` +
`GeminiLiveAudioModel` instead of a text LLM. Before this, Track 2's only
evidence was a hand-run 3-turn demo (a fact stated in turn 1, correctly
recalled in turn 3 -- see `BUGS.md`) -- real and genuinely confirmed
memory *conditioning*, but not a number comparable to anything else in
this project. This section produces that number.

**Doesn't need the GPU** -- both the audio model (Gemini Live API) and
the extraction model (Gemini text API) are API-based, same reasoning as
`llm/gemini_client.py`'s and `gemini_live_model.py`'s own docstrings.
Running it in this same notebook is purely for having one file with every
new result in it, not because it needs anything Section 1-6 set up
(other than the repo clone).

**Does need chromadb/networkx, unlike Section 5's WeightEdit path.**
`HippoAudioPipeline` (`pipeline_audio2audio.py`) genuinely uses
`memory/store.py`'s `HippoMemory` for real -- this isn't the same
throwaway-import situation `run_locomo` had for the WeightEdit baseline
(see Section 2's fix above); Track 2's whole memory architecture is built
on it. Confirmed missing on a real run:
`ModuleNotFoundError: No module named 'chromadb'` from
`gemini_live_model.py`'s own import chain, before ever reaching the audio
model itself.

**A second, related issue, flagged as best-effort rather than confirmed
fixed**: installing chromadb pulls in its own `opentelemetry-*`
dependencies, which on a real run left `opentelemetry.context` (from
`opentelemetry-api`) out of sync with a newer `opentelemetry-sdk` --
`ImportError: cannot import name '_ON_EMIT_RECURSION_COUNT_KEY' from
'opentelemetry.context'`, from deep inside `import chromadb` itself. Same
underlying pattern as Section 2's numpy fix: something earlier in this
same kernel process (Sections 2-6's imports) already had an
`opentelemetry` submodule cached in `sys.modules` before chromadb's
install changed what's on disk, so the next `import chromadb` picked up
a stale cached module instead of the freshly-installed one. Purging any
already-cached `opentelemetry`/`chromadb` modules right after installing
should force a clean re-import -- reasoned from the same confirmed root
cause as the numpy issue, but not itself verified on a real run yet, so
if this specific step still fails, that's the next thing to look at.

In [ ]:
with step('install pyttsx3, espeak, chromadb, networkx'):
    # Same as colab.ipynb's Step 1 comment: pyttsx3 shells out to espeak on
    # Linux, and Colab/Kaggle containers don't have it preinstalled.
    run_shell('apt-get -qq install -y espeak-ng espeak')
    run_shell('pip install -q pyttsx3 soundfile')

    import importlib.util
    # Loose version constraints (matching requirements.txt), not exact pins
    # -- Section 2's chromadb/networkx equivalent isn't needed here since
    # neither Kaggle's base image nor EasyEdit's own install ships these,
    # so there's no already-installed version to accidentally corrupt by
    # reinstalling (see Section 2's fix for why that distinction matters).
    for pkg, import_name in [('chromadb>=0.5.0', 'chromadb'), ('networkx>=3.2', 'networkx')]:
        if importlib.util.find_spec(import_name) is None:
            run_shell(f'pip install -q "{pkg}"')
        else:
            log(f'  {import_name} already importable, skipping')

    # Best-effort fix for a real, confirmed ImportError (see markdown above):
    # purge any opentelemetry/chromadb submodules already cached in
    # sys.modules from earlier in this same kernel process, so the next
    # `import chromadb` (in Section 7b, via pipeline_audio2audio.py) reads
    # fresh from what was just installed instead of a stale cached version.
    import sys
    _purged = [m for m in list(sys.modules) if m == 'opentelemetry' or m.startswith('opentelemetry.')
               or m == 'chromadb' or m.startswith('chromadb.')]
    for m in _purged:
        del sys.modules[m]
    if _purged:
        log(f'Purged {len(_purged)} cached opentelemetry/chromadb submodule(s) before first real import')

## 7b. Load the real Gemini Live audio model

Loaded once, shared across every conversation in Section 8 below --
`GeminiLiveAudioModel` holds no conversation state of its own (every
`respond()` call is an independent `live.connect()` session, confirmed
directly from its own source), so unlike GPT-2 XL in Section 3 there is
no per-conversation reset needed here at all; only each conversation's
`HippoAudioPipeline` (and therefore its memory) needs to be fresh, and
`run_locomo_audio` already does that internally.

Reuses `GEMINI_API_KEY` and `extraction_llm` from Sections 4/4 above --
run those cells first if you skipped straight here.

In [ ]:
with step('load GeminiLiveAudioModel'):
    from gemini_live_model import GeminiLiveAudioModel

    audio_model = GeminiLiveAudioModel()
    audio_model.load()
    log(f'Loaded audio model: {audio_model.model}')

## 7c. Cheap sanity check -- confirm the real audio round trip works before benchmarking

Same discipline as Section 3b: one real, cheap call before spending time
on a multi-conversation run. This is the exact call shape already
confirmed working live during this project's own development (see
`gemini_live_model.py`'s module docstring and
`tests/test_gemini_live_model.py::test_real_live_api_round_trip`) --
re-running it here is about confirming *this* Kaggle session's network/
API key/quota are actually fine, not re-discovering whether the adapter
itself works.

In [ ]:
with step('single-turn audio sanity check'):
    require('audio_model')
    import tempfile
    from tts.model import load_tts
    from tts.synthesize import synthesize

    _sanity_audio = tempfile.mktemp(suffix='.wav')
    synthesize(load_tts(), 'What is two plus two?', _sanity_audio)
    _, _sanity_transcript = audio_model.respond(_sanity_audio)
    log(f'Sanity check reply: {_sanity_transcript!r}')

print()
print('>>> Check the reply above -- it should be a real, on-topic spoken answer')
print('    (e.g. mentioning "four"). If it is empty or nonsensical, stop and')
print('    debug network/API-key/quota issues before running Section 8.')

## 8. Run the real LoCoMo-style audio benchmark

Ingestion (`ingest_text_turn` per turn) is pure text -- same cost profile
as any other baseline's ingestion, since a benchmark replaying a
transcript for extraction has no reason to synthesize and "speak" all
369-689 turns just to store them (see `evaluate_audio.py`'s module
docstring). Only the QA step pays audio-specific cost: each question is
TTS-synthesized and sent through a real Gemini Live round trip.

**Scope starts deliberately small for the same reason as Section 5.**
`NUM_CONVERSATIONS=1`, `MAX_QA_PER_CONVERSATION=10` first -- widen only
after seeing the printed wall-clock reading.

In [ ]:
from benchmarks.locomo.evaluate_audio import run_locomo_audio

AUDIO_NUM_CONVERSATIONS = 1
AUDIO_MAX_QA_PER_CONVERSATION = 10

AUDIO_CHECKPOINT_PATH = (
    ('/kaggle/working' if os.path.exists('/kaggle/working') else '/content')
    + '/locomo_checkpoint_hippoaudio.json'
)

audio_result = None
audio_elapsed = None

with step(f'run_locomo_audio (HippoAudio, {AUDIO_NUM_CONVERSATIONS} conv, '
          f'{AUDIO_MAX_QA_PER_CONVERSATION} qa/conv)'):
    require('audio_model', 'extraction_llm')
    log(f'Checkpoint path: {AUDIO_CHECKPOINT_PATH}')
    _t0 = time.time()
    audio_result = run_locomo_audio(
        llm_client=extraction_llm,
        audio_model=audio_model,
        num_conversations=AUDIO_NUM_CONVERSATIONS,
        max_qa_per_conversation=AUDIO_MAX_QA_PER_CONVERSATION,
        checkpoint_path=AUDIO_CHECKPOINT_PATH,
    )
    audio_elapsed = time.time() - _t0

    log(f"avg F1: {audio_result['avg_f1']:.1%}  (over {audio_result['total']} questions)  "
        f"bins: {audio_result['bins']}  wall_clock: {audio_elapsed:.0f}s")


# Guarded the same way as Section 5's equivalent block -- see its comment.
if audio_result is not None:
    print('=' * 60)
    print(f"HippoAudio LoCoMo avg F1: {audio_result['avg_f1']:.1%}  "
          f"(over {audio_result['total']} questions)")
    print(f"bins: {audio_result['bins']}")
    print(f'Wall clock: {audio_elapsed:.0f}s for {AUDIO_NUM_CONVERSATIONS} conversation(s) '
          f'-- use this to size AUDIO_NUM_CONVERSATIONS/AUDIO_MAX_QA_PER_CONVERSATION for a bigger run.')
    print('=' * 60)
    for d in sorted(audio_result['details'], key=lambda d: -d['f1'])[:10]:
        print(f"  f1={d['f1']:.2f} cat={d['category']}  Q: {d['question']}")
        print(f"    gold={d['gold']!r}  predicted={d['predicted']!r}")
else:
    print(f'run_locomo_audio did not complete -- see {LOG_PATH} for why. Section 9 will still save a results file.')

## 9. Save Track 2 audio results (intermediate -- combined with WeightEdit above in Section 10)

In [ ]:
with step('save HippoAudio results'):
    AUDIO_RESULTS_PATH = (
        ('/kaggle/working' if os.path.exists('/kaggle/working') else '/content')
        + '/hippovoice_results_audio.json'
    )

    audio_out = {
        'timestamp': datetime.datetime.now().isoformat(),
        'system': 'HippoAudio',
        'audio_model': getattr(audio_model, 'model', 'not reached') if 'audio_model' in dir() else 'not reached',
        'extraction_llm': extraction_llm.model_name if 'extraction_llm' in dir() else 'not reached',
        'num_conversations': globals().get('AUDIO_NUM_CONVERSATIONS', 'not reached'),
        'max_qa_per_conversation': globals().get('AUDIO_MAX_QA_PER_CONVERSATION', 'not reached'),
        'wall_clock_seconds': audio_elapsed if 'audio_result' in dir() and audio_result is not None else 'not reached',
        'locomo': (
            {
                'avg_f1': audio_result['avg_f1'],
                'total': audio_result['total'],
                'bins': audio_result['bins'],
            }
            if 'audio_result' in dir() and audio_result is not None
            else 'not reached -- see run_log.txt for where the run stopped'
        ),
    }

    with open(AUDIO_RESULTS_PATH, 'w') as f:
        json.dump(audio_out, f, indent=2)

    log(f'Saved to {AUDIO_RESULTS_PATH}')

if 'audio_out' in dir():
    print(f'Results: {AUDIO_RESULTS_PATH}')
    print(json.dumps(audio_out, indent=2))
else:
    print(f'Saving HippoAudio results failed -- see {LOG_PATH}.')

## 10. Save combined results + full comparison summary

Everything this notebook produced, in one file, next to the numbers that
already exist from `colab.ipynb` (hardcoded below since they don't change
between runs of *this* notebook -- re-running them isn't needed unless
the underlying code for those systems changes; see `BUGS.md` for their
provenance). This is the closest thing to "the full picture" without
literally re-running every system from scratch: five real, F1-scored
LoCoMo numbers on the same dataset and scoring methodology, side by side.

**Still not the whole live-demo picture** -- this table is retrieval/
memory-architecture comparison, not a recording of an actual spoken
conversation. For that, the closest thing is the local 3-turn
`run_hippoaudio_demo.py`-style transcript already captured in this
project's history (a fact stated in turn 1 correctly recalled in turn 3),
which this notebook doesn't reproduce here since it needs a live
microphone-less TTS setup identical to what Section 7/8 already did --
Section 8's own per-question `predicted`/`gold` pairs in `run_log.txt`
are that same kind of evidence, just for held-out questions instead of a
scripted follow-up.

Ends by raising if anything in `_failures` is non-empty, so the Kaggle
commit is still correctly marked failed when something broke -- but only
after every artifact that could be saved, was.

In [ ]:
FULL_RESULTS_PATH = (
    ('/kaggle/working' if os.path.exists('/kaggle/working') else '/content')
    + '/hippovoice_full_benchmark_results.json'
)

combined_out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'new_this_run': {
        'weightedit': globals().get('weightedit_out', 'not reached'),
        'hippoaudio': globals().get('audio_out', 'not reached'),
    },
    # From colab.ipynb (2026-07-11 commit, 10 conversations, 1540 questions
    # for HippoVoice; see BUGS.md for full provenance) -- not re-run here.
    'existing_from_colab_ipynb': {
        'HippoVoice':  {'avg_f1': 0.241, 'total': 1540, 'note': '10 conversations, all QA pairs'},
        'Mem0-style':  {'avg_f1': 'see hippovoice_results_mem0.json from that run'},
        'AMem-style':  {'avg_f1': 'see hippovoice_results_amem.json from that run'},
        'NaiveRAG':    {'avg_f1': 'see hippovoice_results_naive.json from that run'},
    },
    'failures_this_run': [{'step': name, 'error': err} for name, err in _failures],
}

with open(FULL_RESULTS_PATH, 'w') as f:
    json.dump(combined_out, f, indent=2)

log(f'Saved combined results to {FULL_RESULTS_PATH}')
print(f'Full run log: {LOG_PATH}')
print(json.dumps(combined_out, indent=2))

if _failures:
    raise RuntimeError(
        f'{len(_failures)} step(s) failed this run: {[name for name, _ in _failures]} -- '
        f'see {LOG_PATH} for full tracebacks. Every artifact that could be saved, was, '
        f'before raising this so the commit is still correctly marked failed.'
    )